In [1]:
import numpy as np
import GPy

In [13]:
X = np.linspace(0, 10, 20)[:, None]  # 20 points in 1D
Y = np.sin(X) + 0.1 * np.random.randn(*X.shape)  # Noisy observations

# Define a kernel (RBF + Gaussian noise)
kernel = GPy.kern.RBF(input_dim=1, name="rbf")
# Create GP regression model
gp_model = GPy.models.GPRegression(X, Y, kernel)

In [14]:
gp_model.kern.lengthscale.values[0]

1.0

In [2]:
import numpy as np
import sympy

# Define the length scale symbol
l = sympy.symbols(r"\theta_l")
x1 = sympy.symbols(r"x_1")
x2 = sympy.symbols(r"x_2")
x3 = sympy.symbols(r"x_3")
y1 = sympy.symbols(r"y_1")
y2 = sympy.symbols(r"y_2")
y3 = sympy.symbols(r"y_3")
mu1 = sympy.symbols(r"\mu_1")
mu2 = sympy.symbols(r"\mu_2")
mu3 = sympy.symbols(r"\mu_3")
mu = sympy.symbols(r"\mu")
sigma = sympy.symbols(r"\sigma")
sigma_n = sympy.symbols(r"\sigma_n")

# Define the kernel function
func = lambda x,y: sympy.exp(- ((x-y)**2)/ (2*l))


x = sympy.Matrix([x1, x2])
y = sympy.Matrix([y1, y2])
mu = sympy.Matrix([mu, mu])


Sigma = [[func(xi, xj) for xj in x] for xi in x]
Sigma = sympy.Matrix(Sigma)

N = sigma_n * sympy.eye(len(x))


Cov = Sigma + N
Cov_inv = Cov.inv()
Cov_log_det_m = sympy.Matrix([sympy.log(Cov.det())])
Cov_det = Cov.det()
# Define the likelihood function and log likelihood function
likelihood = sympy.exp(-0.5 * (mu - y).T * Cov_inv * (mu - y)) / sympy.sqrt((2*sympy.pi)**2 * Cov_det)
log_likelihood = sympy.log(likelihood)
# Manually defining the log likelihood function again since smypy is making rather large expressions
mll = (-1)* (1/2 * (mu - y).T * Cov_inv * (mu - y) + 1/2 * Cov_log_det_m + 1/2 * 2 * sympy.Matrix([sympy.log(2*sympy.pi)]))
mll

Matrix([[-(\mu - y_1)*((0.5*\mu - 0.5*y_1)*(\sigma_n + 1)/(\sigma_n**2 + 2*\sigma_n - exp(-x_1**2/\theta_l + 2*x_1*x_2/\theta_l - x_2**2/\theta_l) + 1) - (0.5*\mu - 0.5*y_2)*exp(-x_1**2/(2*\theta_l) + x_1*x_2/\theta_l - x_2**2/(2*\theta_l))/(\sigma_n**2 + 2*\sigma_n - exp(-x_1**2/\theta_l + 2*x_1*x_2/\theta_l - x_2**2/\theta_l) + 1)) - (\mu - y_2)*(-(0.5*\mu - 0.5*y_1)*exp(-x_1**2/\theta_l + 2*x_1*x_2/\theta_l - x_2**2/\theta_l)/(\sigma_n**2*exp(-x_1**2/(2*\theta_l) + x_1*x_2/\theta_l - x_2**2/(2*\theta_l)) + 2*\sigma_n*exp(-x_1**2/(2*\theta_l) + x_1*x_2/\theta_l - x_2**2/(2*\theta_l)) - exp(-x_1**2/\theta_l + 2*x_1*x_2/\theta_l - x_2**2/\theta_l)*exp(-x_1**2/(2*\theta_l) + x_1*x_2/\theta_l - x_2**2/(2*\theta_l)) + exp(-x_1**2/(2*\theta_l) + x_1*x_2/\theta_l - x_2**2/(2*\theta_l))) + (0.5*\mu - 0.5*y_2)*(\sigma_n + 1)/(\sigma_n**2 + 2*\sigma_n - exp(-x_1**2/\theta_l + 2*x_1*x_2/\theta_l - x_2**2/\theta_l) + 1)) - 0.5*log(\sigma_n**2 + 2*\sigma_n - exp(-x_1**2/\theta_l + 2*x_1*x_2/\thet

In [4]:
# differentiting with respect to the lengthscale hyperparameter
mll_diff = sympy.diff(mll, l, sigma, sigma_n, mu)
# Simplify the expression (Can take a while)
# mll_diff = sympy.simplify(mll_diff)
# mll_diff
# eq = sympy.Eq(mll_diff[0], 0)
# eq

In [5]:
mll_diff

Matrix([[0]])